This script does...
- Take cleaned dataset
- Perform double selection to select image features X
- Perform 2SLS

In [1]:
import sys
print(sys.executable)

/opt/anaconda3/envs/mosaiks-env/bin/python


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
import os

os.chdir('/Users/ryotarohiraki/Desktop/Spring 2026/Capstone/projects')


In [2]:
cps14 = pd.read_parquet("dataset/cleaned_dataset/cleaned_cps14_with_nearest_college_imgfeat.parquet")
# cps13 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps13_with_nearest_college_imgfeat.parquet')
# cps12 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps12_with_nearest_college_imgfeat.parquet')

cps14.columns

Index(['person_num', 'log_wage', 'educ', 'exp', 'exp2', 'female', 'black',
       'mv', 'age', 'birth_place',
       ...
       'mosaiks_3990', 'mosaiks_3991', 'mosaiks_3992', 'mosaiks_3993',
       'mosaiks_3994', 'mosaiks_3995', 'mosaiks_3996', 'mosaiks_3997',
       'mosaiks_3998', 'mosaiks_3999'],
      dtype='object', length=4024)

In [3]:
cps14["STATE_PUMA"].head()

0    01_00501
1    01_01801
2    01_01404
3    01_00100
4    01_00100
Name: STATE_PUMA, dtype: object

In [3]:
# Double Selection + IV (without weights and with fixed effects)
# 1) Lasso: y ~ X (after partialling out W + FE)
# 2) Lasso: d ~ X (after partialling out W + FE)
# 3) Lasso: z ~ X (after partialling out W + FE)
# 4) 2SLS: y ~ d + selected_X + W + FE, instrument d by z

def _build_controls(df, w_cols=None, fe_cols=None):
    w_cols = [] if w_cols is None else list(w_cols)
    fe_cols = [] if fe_cols is None else list(fe_cols)

    parts = []
    if len(w_cols) > 0:
        parts.append(df[w_cols].astype(float))

    if len(fe_cols) > 0:
        fe = df[fe_cols].copy()
        for c in fe_cols:
            fe[c] = fe[c].astype('category')
        fe_dummies = pd.get_dummies(fe, drop_first=True, dtype=float)
        parts.append(fe_dummies)

    if len(parts) == 0:
        return pd.DataFrame(index=df.index)

    out = pd.concat(parts, axis=1)
    out = out.loc[:, ~out.columns.duplicated()]
    return out


def _partial_out(v, controls, weights=None):
    y = np.asarray(v, dtype=float)
    if controls.shape[1] == 0:
        return y

    Xc = sm.add_constant(controls, has_constant='add')
    if weights is None:
        fit = sm.OLS(y, Xc).fit()
    else:
        fit = sm.WLS(y, Xc, weights=weights).fit()
    return fit.resid

def _partial_out_matrix(V, controls, weights=None):
    V = np.asarray(V, dtype=float)
    if controls.shape[1] == 0:
        return V

    Xc = sm.add_constant(controls, has_constant='add')
    if weights is None:
        fit = sm.OLS(V, Xc).fit()
    else:
        fit = sm.WLS(V, Xc, weights=weights).fit()
    return fit.resid  # shape (n, p)


def _safe_standardize(X, weights=None):
    scaler = StandardScaler()
    if weights is None:
        return scaler.fit_transform(X)
    try:
        return scaler.fit_transform(X, sample_weight=weights)
    except TypeError:
        return scaler.fit_transform(X)


def double_selection_iv(
    df,
    outcome_col,
    treatment_col,
    instrument_col,
    x_cols,
    w_cols=None,
    fe_cols=None,
    weight_col=None,
    cluster_col=None,
    cv=5,
    random_state=42,
    max_iter=100000,
    cov_type='clustered',
    tol=1e-3
):
    x_cols = list(x_cols)
    w_cols = [] if w_cols is None else list(w_cols)
    fe_cols = [] if fe_cols is None else list(fe_cols)

    use_cols = [outcome_col, treatment_col, instrument_col] + x_cols + w_cols + fe_cols
    if weight_col is not None:
        use_cols = use_cols + [weight_col]
    if cluster_col is not None:
        use_cols = use_cols + [cluster_col]

    dat = df[use_cols].dropna().copy()
    if weight_col is not None:
        dat = dat[pd.to_numeric(dat[weight_col], errors='coerce') > 0].copy()

    y = dat[outcome_col].astype(float).to_numpy()
    d = dat[treatment_col].astype(float).to_numpy()
    z = dat[instrument_col].astype(float).to_numpy()
    X = dat[x_cols].astype(float)
    weights = dat[weight_col].astype(float).to_numpy() if weight_col is not None else None

    controls = _build_controls(dat, w_cols=w_cols, fe_cols=fe_cols)

    y_tilde = _partial_out(y, controls)
    d_tilde = _partial_out(d, controls)
    z_tilde = _partial_out(z, controls)

    X_tilde = _partial_out_matrix(X, controls)
    X_std = _safe_standardize(X_tilde)

    lasso_y = LassoCV(cv=cv, random_state=random_state, max_iter=max_iter, tol=tol, selection="random").fit(X_std, y_tilde)
    lasso_d = LassoCV(cv=cv, random_state=random_state, max_iter=max_iter, tol=tol, selection="random").fit(X_std, d_tilde)
    lasso_z = LassoCV(cv=cv, random_state=random_state, max_iter=max_iter, tol=tol, selection="random").fit(X_std, z_tilde)

    sel_y = set(np.where(np.abs(lasso_y.coef_) > 1e-12)[0].tolist())
    sel_d = set(np.where(np.abs(lasso_d.coef_) > 1e-12)[0].tolist())
    sel_z = set(np.where(np.abs(lasso_z.coef_) > 1e-12)[0].tolist())
    sel_union_idx = sorted(sel_y.union(sel_d).union(sel_z))
    selected_x = [x_cols[i] for i in sel_union_idx]

    exog_parts = []
    if len(selected_x) > 0:
        exog_parts.append(dat[selected_x].astype(float))
    if len(w_cols) > 0:
        exog_parts.append(dat[w_cols].astype(float))
    if len(fe_cols) > 0:
        exog_parts.append(pd.get_dummies(dat[fe_cols].astype('category'), drop_first=True, dtype=float))

    if len(exog_parts) > 0:
        exog_base = pd.concat(exog_parts, axis=1)
        exog_base = exog_base.loc[:, ~exog_base.columns.duplicated()]
    else:
        exog_base = pd.DataFrame(index=dat.index)

    exog = sm.add_constant(exog_base, has_constant='add')
    endog = dat[[treatment_col]].astype(float)
    instr = dat[[instrument_col]].astype(float)

    iv_model = IV2SLS(
        dependent=dat[outcome_col].astype(float),
        exog=exog,
        endog=endog,
        instruments=instr,
        weights=weights
    ).fit(cov_type=cov_type, clusters=dat[cluster_col])

    return {
        'n_obs': int(len(dat)),
        'selected_x': selected_x,
        'n_selected_x': int(len(selected_x)),
        'alpha_y': float(lasso_y.alpha_),
        'alpha_d': float(lasso_d.alpha_),
        'alpha_z': float(lasso_z.alpha_),
        'n_sel_y': int(len(sel_y)),
        'n_sel_d': int(len(sel_d)),
        'n_sel_z': int(len(sel_z)),
        'iv_2sls': iv_model,
        "first_stage": iv_model.first_stage
    }


# ====== Fill variables here ======
DATA = cps14.copy()  # e.g. cps12 / cps13 / cps14
OUTCOME_COL = 'log_wage'    # e.g. 'log_wage'
TREATMENT_COL = 'educ'  # e.g. 'educ'
INSTRUMENT_COL = 'nearest_college_dist_km' # e.g. 'nearest_college_dist_km'
X_COLS = [c for c in DATA.columns if c.startswith('mosaiks_')]         # e.g. [c for c in DATA.columns if c.startswith('img_mosaiks_')]
W_COLS = ['exp','exp2', "female", "black"]           # e.g. ['exp', 'exp2', 'female', 'black']
FE_COLS = []          # e.g. ['state', 'year']
WEIGHT_COL = 'pums_weight'     # e.g. 'pums_weight'

if OUTCOME_COL is None or TREATMENT_COL is None or INSTRUMENT_COL is None or X_COLS is None:
    raise ValueError('Set OUTCOME_COL, TREATMENT_COL, INSTRUMENT_COL, X_COLS before running.')

ds_res = double_selection_iv(
    df=DATA,
    outcome_col=OUTCOME_COL,
    treatment_col=TREATMENT_COL,
    instrument_col=INSTRUMENT_COL,
    x_cols=X_COLS,
    w_cols=W_COLS,
    #fe_cols=FE_COLS,
    weight_col=WEIGHT_COL,
    cluster_col="STATE_PUMA"
)

print('n_obs:', ds_res['n_obs'])
print('n_sel_y:', ds_res['n_sel_y'])
print('n_sel_d:', ds_res['n_sel_d'])
print('n_sel_z:', ds_res['n_sel_z'])
print('n_selected_x:', ds_res['n_selected_x'])
print('first selected_x:', ds_res['selected_x'][:20])
print(ds_res["first_stage"])
display(ds_res['iv_2sls'].summary)


n_obs: 146638
n_sel_y: 71
n_sel_d: 53
n_sel_z: 99
n_selected_x: 195
first selected_x: ['mosaiks_2', 'mosaiks_10', 'mosaiks_15', 'mosaiks_39', 'mosaiks_41', 'mosaiks_46', 'mosaiks_48', 'mosaiks_54', 'mosaiks_55', 'mosaiks_59', 'mosaiks_75', 'mosaiks_77', 'mosaiks_129', 'mosaiks_142', 'mosaiks_177', 'mosaiks_197', 'mosaiks_208', 'mosaiks_220', 'mosaiks_232', 'mosaiks_267']
     First Stage Estimation Results    
                                   educ
---------------------------------------
R-squared                        0.1046
Partial R-squared                0.0007
Shea's R-squared                 0.0007
Partial F-statistic              39.316
P-value (Partial F-stat)      3.605e-10
Partial F-stat Distn            chi2(1)
========================== ============
const                            6.9314
                               (2.6275)
mosaiks_2                        218.56
                               (0.7161)
mosaiks_10                       362.80
                          

<class 'linearmodels.compat.statsmodels.Summary'>
"""
                          IV-2SLS Estimation Summary                          
==============================================================================
Dep. Variable:               log_wage   R-squared:                      0.0613
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0600
No. Observations:              146638   F-statistic:                 1.892e+11
Date:                Sat, Apr 04 2026   P-value (F-stat)                0.0000
Time:                        03:51:16   Distribution:                chi2(200)
Cov. Estimator:             clustered                                         
                                                                              
                              Parameter Estimates                               
================================================================================
              Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------
const            14.053     0.7098     19.799     0.0000      12.662      15.444
mosaiks_2       -4.0084     67.250    -0.0596     0.9525     -135.82      127.80
mosaiks_10       96.639     408.35     0.2367     0.8129     -703.71      896.99
mosaiks_15      -12.449     15.149    -0.8217     0.4112     -42.141      17.244
mosaiks_39      -8652.7  1.643e+04    -0.5267     0.5984  -4.085e+04   2.355e+04
mosaiks_41       0.0351     0.1659     0.2117     0.8323     -0.2900      0.3602
mosaiks_46       13.049     69.853     0.1868     0.8518     -123.86      149.96
mosaiks_48       21.722     22.045     0.9854     0.3244     -21.485      64.930
mosaiks_54       0.1527     0.2006     0.7612     0.4465     -0.2404      0.5458
mosaiks_55      -1.3143     1.5790    -0.8323     0.4052     -4.4091      1.7806
mosaiks_59      -0.9031     0.8970    -1.0068     0.3140     -2.6611      0.8549
mosaiks_75      -11.464     23.866    -0.4804     0.6310     -58.240      35.312
mosaiks_77      -4913.0     5886.2    -0.8347     0.4039  -1.645e+04      6623.7
mosaiks_129     -78.663     637.54    -0.1234     0.9018     -1328.2      1170.9
mosaiks_142      221.27     837.17     0.2643     0.7915     -1419.6      1862.1
mosaiks_177      20.440     17.443     1.1718     0.2413     -13.748      54.628
mosaiks_197      103.12     66.595     1.5485     0.1215     -27.399      233.65
mosaiks_208     -17.636     25.764    -0.6845     0.4937     -68.132      32.861
mosaiks_220     -2.2110     3.5332    -0.6258     0.5315     -9.1359      4.7139
mosaiks_232     -0.1032     0.2601    -0.3968     0.6915     -0.6130      0.4066
mosaiks_267     -27.439     18.835    -1.4568     0.1452     -64.356      9.4779
mosaiks_273      1.0837     3.1873     0.3400     0.7339     -5.1633      7.3307
mosaiks_318     -97.376     98.286    -0.9907     0.3218     -290.01      95.262
mosaiks_351      4.7443     2.8174     1.6839     0.0922     -0.7778      10.266
mosaiks_365     -0.1974     0.1694    -1.1655     0.2438     -0.5294      0.1346
mosaiks_367     -31.007     163.33    -0.1898     0.8494     -351.14      289.12
mosaiks_393      314.35     163.66     1.9208     0.0548     -6.4154      635.12
mosaiks_407      121.03     69.151     1.7502     0.0801     -14.504      256.56
mosaiks_409      0.3135     1.3191     0.2376     0.8122     -2.2719      2.8988
mosaiks_415      792.72     1578.7     0.5021     0.6156     -2301.6      3887.0
mosaiks_419      0.2060     0.5509     0.3740     0.7084     -0.8737      1.2857
mosaiks_426     -0.3706     0.6511    -0.5693     0.5692     -1.6467      0.9054
mosaiks_432     -103.31     78.131    -1.3222     0.1861     -256.44      49.827
mosaiks_450     -16.294     136.93    -0.1190     0.9053     -284.67      252.08
mosaiks_489     -99.878     53.276    -1.8747     0.0608     -204.30      4.5414
mosaiks_513     -318.89     443.81    -0.7185     0.4724     -1188.7      55